In [1]:
!pip install git+https://github.com/p-priyanshu04/Deep-Learning-Project.git

  Cloning https://github.com/p-priyanshu04/Deep-Learning-Project.git to /tmp/pip-req-build-nzme_fhq
  Running command git clone --filter=blob:none --quiet https://github.com/p-priyanshu04/Deep-Learning-Project.git /tmp/pip-req-build-nzme_fhq
  Resolved https://github.com/p-priyanshu04/Deep-Learning-Project.git to commit 304b20f2e069d387735d4b5f320854f9128bbaca
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for deep-learning-project: filename=deep_learning_project-0.1.0-py3-none-any.whl size=8522 sha256=926e877f3d954cb0ca3e98d04af2f8602ab1494a9d2705eca7e55e351aeff40a
  Stored in directory: /tmp/pip-ephem-wheel-cache-e2xf434e/wheels/b9/5a/e9/64c42444830fcdb95c62aadced304cd0453e3afd31e9493ee7
Successfully built deep-learning-project


# Data Preprocessing

This notebook demonstrates loading the WikiQA and TREC-QA datasets, and running our preprocessing pipeline.

In [2]:
import sys
sys.path.append('..')

from datasets import load_dataset
from utils.data_utils import *
import torch
from torch.utils.data import DataLoader

# Environment setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cpu


### 1. Load Datasets
We load WikiQA and TREC-QA from HuggingFace.

In [3]:
wiki_qa = load_dataset('wiki_qa')
trec_qa_raw = load_dataset('lucadiliello/trecqa')

print('WikiQA splits:', list(wiki_qa.keys()))
print('TREC-QA splits:', list(trec_qa_raw.keys()))

README.md: 0.00B [00:00, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/594k [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/264k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/2.00M [00:00<?, ?B/s]

Generating test split:   0%|          | 0/6165 [00:00<?, ? examples/s]

...

Generating dev split:   0%|          | 0/1364 [00:00<?, ? examples/s]

Generating dev_clean split:   0%|          | 0/1343 [00:00<?, ? examples/s]

Generating test_clean split:   0%|          | 0/1442 [00:00<?, ? examples/s]

Generating train_all split:   0%|          | 0/53417 [00:00<?, ? examples/s]

WikiQA splits: ['test', 'validation', 'train']
TREC-QA splits: ['train', 'test', 'dev', 'dev_clean', 'test_clean', 'train_all']


### 2. Build Vocabulary & Embeddings
We build a vocabulary from the training sets of both datasets, then load GloVe embeddings.

In [4]:
word2idx, idx2word = build_vocab(wiki_qa['train'], trec_qa_raw['train'])
print(f'Vocabulary size: {len(word2idx)}')

download_glove()
glove_vectors = load_glove()
embedding_matrix = build_embedding_matrix(word2idx, glove_vectors)

Vocabulary size: 40428
Extracting...
Done.
Loaded 400,000 GloVe vectors.
GloVe coverage: 35294/40428 (87.3%)


### 3. Create PyTorch Datasets
We instantiate the `QADataset` which handles tokenization, padding, and finding question word positions in the answers.

In [5]:
BATCH_SIZE = 64

# WikiQA
wiki_train_ds = QADataset(wiki_qa['train'], word2idx)
wiki_train_loader = DataLoader(wiki_train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)

print(f'WikiQA Train size: {len(wiki_train_ds)}')

# Display sample
sample = wiki_train_ds[0]
print('\nSample Keys:', sample.keys())
print('Question Length:', sample['q_len'])
print('Answer Length:', sample['a_len'])
print('Label:', sample['label'].item())

WikiQA Train size: 20360

Sample Keys: dict_keys(['q_ids', 'a_ids', 'q_len', 'a_len', 'q_pos', 'label', 'qid'])
Question Length: 6
Answer Length: 10
Label: 0.0
